# LigaInsider Bundesliga Predicted Lineups

This notebook loads each supplied Bundesliga team page through one undetected Chrome 150 session, extracts the predicted formation and candidate alternatives, then writes complete home/away fixtures as a JSON snapshot.

LigaInsider emits formation columns from right to left. Player alternatives (`sub_child`) are preserved in their displayed probability order.

## 1. Resolve project paths

In [1]:
# Import the libraries required by this notebook step.
import sys
from pathlib import Path


# Handle project root for reuse in the workflow.
def locate_project_root() -> Path:
    starts = []
    vscode_notebook = globals().get('__vsc_ipynb_file__')
    if isinstance(vscode_notebook, str) and vscode_notebook.strip():
        starts.append(Path(vscode_notebook).expanduser().resolve().parent)
    starts.append(Path.cwd().resolve())
    checked = set()
    # Process each available item while preserving the current workflow state.
    for start in starts:
        # Process each available item while preserving the current workflow state.
        for candidate in (start, *start.parents):
            if candidate in checked:
                continue
            checked.add(candidate)
            if (candidate / 'project_paths.py').is_file():
                return candidate
    raise FileNotFoundError('Could not locate project_paths.py.')


# Set workflow configuration value: PROJECT_ROOT.
PROJECT_ROOT = locate_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from project_paths import LIGAINSIDER_PREDICTED_LINEUPS_DIR, ensure_directory

output_directory = ensure_directory(LIGAINSIDER_PREDICTED_LINEUPS_DIR)
print(f'LigaInsider lineup output directory: {output_directory}')

LigaInsider lineup output directory: C:\kickbase project\outputs\ligainsider\predicted_lineups


## 2. Imports and source configuration

In [2]:
# Import the libraries required by this notebook step.
import json
import re
from datetime import datetime
from typing import Any
from urllib.parse import unquote, urljoin, urlparse
from zoneinfo import ZoneInfo

# Handle expected failures with a clear, actionable message.
try:
    import undetected_chromedriver as uc
    from bs4 import BeautifulSoup
    from selenium.common.exceptions import TimeoutException, WebDriverException
    from selenium.webdriver.common.by import By
    from selenium.webdriver.support import expected_conditions as EC
    from selenium.webdriver.support.ui import WebDriverWait
except ImportError as exc:
    raise ImportError(
        'Required packages are missing. Install them with: '
        '%pip install beautifulsoup4 selenium undetected-chromedriver'
    ) from exc

# Set workflow configuration value: SOURCE_NAME.
SOURCE_NAME = 'LigaInsider'
# Set workflow configuration value: SOURCE_URL.
SOURCE_URL = 'https://www.ligainsider.de/'
# Set workflow configuration value: CHROME_MAJOR_VERSION.
CHROME_MAJOR_VERSION = 150
# Set workflow configuration value: PAGE_LOAD_TIMEOUT_SECONDS.
PAGE_LOAD_TIMEOUT_SECONDS = 45
# Set workflow configuration value: WAIT_TIMEOUT_SECONDS.
WAIT_TIMEOUT_SECONDS = 30
# Set workflow configuration value: REQUIRED_HEADING.
REQUIRED_HEADING = 'VORAUSSICHTLICHE AUFSTELLUNG'
# Set workflow configuration value: BERLIN_TIMEZONE.
BERLIN_TIMEZONE = ZoneInfo('Europe/Berlin')

# Set workflow configuration value: TEAM_PAGES.
TEAM_PAGES = {
    'Bayern München': 'https://www.ligainsider.de/fc-bayern-muenchen/1/',
    'Borussia Dortmund': 'https://www.ligainsider.de/borussia-dortmund/14/',
    'RB Leipzig': 'https://www.ligainsider.de/rb-leipzig/1311/',
    'VfB Stuttgart': 'https://www.ligainsider.de/vfb-stuttgart/12/',
    'TSG Hoffenheim': 'https://www.ligainsider.de/tsg-hoffenheim/10/',
    'Bayer 04 Leverkusen': 'https://www.ligainsider.de/bayer-04-leverkusen/4/',
    'SC Freiburg': 'https://www.ligainsider.de/sc-freiburg/18/',
    'Eintracht Frankfurt': 'https://www.ligainsider.de/eintracht-frankfurt/3/',
    'FC Augsburg': 'https://www.ligainsider.de/fc-augsburg/21/',
    'FSV Mainz 05': 'https://www.ligainsider.de/1-fsv-mainz-05/17/',
    '1. FC Union Berlin': 'https://www.ligainsider.de/1-fc-union-berlin/1246/',
    'Borussia Mönchengladbach': 'https://www.ligainsider.de/borussia-moenchengladbach/5/',
    'Hamburger SV': 'https://www.ligainsider.de/hamburger-sv/9/',
    '1. FC Köln': 'https://www.ligainsider.de/1-fc-koeln/15/',
    'SV Werder Bremen': 'https://www.ligainsider.de/sv-werder-bremen/2/',
    'Schalke 04': 'https://www.ligainsider.de/fc-schalke-04/13/',
    'SV 07 Elversberg': 'https://www.ligainsider.de/sv-07-elversberg/1331/',
    'SC Paderborn': 'https://www.ligainsider.de/sc-paderborn-07/1249/',
}

# Set workflow configuration value: TEAM_ALIASES.
TEAM_ALIASES = {
    'FC Bayern München': 'Bayern München', 'Bayern München': 'Bayern München',
    'Borussia Dortmund': 'Borussia Dortmund', 'RB Leipzig': 'RB Leipzig',
    'VfB Stuttgart': 'VfB Stuttgart', 'TSG Hoffenheim': 'TSG Hoffenheim',
    'Bayer 04 Leverkusen': 'Bayer 04 Leverkusen', 'SC Freiburg': 'SC Freiburg',
    'Eintracht Frankfurt': 'Eintracht Frankfurt', 'FC Augsburg': 'FC Augsburg',
    '1. FSV Mainz 05': 'FSV Mainz 05', 'FSV Mainz 05': 'FSV Mainz 05',
    '1. FC Union Berlin': '1. FC Union Berlin',
    'Borussia Mönchengladbach': 'Borussia Mönchengladbach',
    'Hamburger SV': 'Hamburger SV', '1. FC Köln': '1. FC Köln',
    'SV Werder Bremen': 'SV Werder Bremen', 'FC Schalke 04': 'Schalke 04',
    'Schalke 04': 'Schalke 04', 'SV 07 Elversberg': 'SV 07 Elversberg',
    'SC Paderborn 07': 'SC Paderborn', 'SC Paderborn': 'SC Paderborn',
}


## 3. Parsing helpers and position resolver

In [3]:
# Set workflow configuration value: FIXTURE_PATTERN.
FIXTURE_PATTERN = re.compile(r'^(Heimspiel|Auswärtsspiel)\s+(.+?)\s+gegen\s+(.+)$')
# Set workflow configuration value: DATE_TIME_PATTERN.
DATE_TIME_PATTERN = re.compile(
    r'^(?:Mo|Di|Mi|Do|Fr|Sa|So)\.\s*(\d{2}\.\d{2}\.\d{4})\s*\|\s*(\d{1,2}:\d{2})$'
)


# Clean text for reuse in the workflow.
def clean_text(value: Any) -> str | None:
    if value is None:
        return None
    cleaned = re.sub(r'\s+', ' ', str(value).replace('\xa0', ' ')).strip()
    return cleaned or None


# Handle name from player url for reuse in the workflow.
def full_name_from_player_url(player_url: str) -> str:
    slug = unquote(urlparse(player_url).path.rstrip('/').rsplit('/', 1)[-1])
    slug = re.sub(r'_\d+$', '', slug)
    full_name = clean_text(slug.replace('-', ' ').title())
    # Validate the input before continuing with later processing.
    if full_name is None:
        raise ValueError(f'Could not derive a player name from {player_url!r}.')
    return full_name


# Handle id from player url for reuse in the workflow.
def player_id_from_player_url(player_url: str) -> int:
    slug = urlparse(player_url).path.rstrip('/').rsplit('/', 1)[-1]
    match = re.search(r'_(\d+)$', slug)
    # Validate the input before continuing with later processing.
    if match is None:
        raise ValueError(f'Could not extract a player ID from {player_url!r}.')
    return int(match.group(1))


# Parse and validate fixture text for reuse in the workflow.
def parse_fixture_text(fixture_text: str) -> tuple[str, str, str]:
    fixture_match = FIXTURE_PATTERN.fullmatch(fixture_text)
    # Validate the input before continuing with later processing.
    if fixture_match is None:
        raise ValueError(f'Unexpected fixture text: {fixture_text!r}.')
    german_side, date_time_text, opponent_name = fixture_match.groups()
    date_time_match = DATE_TIME_PATTERN.fullmatch(date_time_text)
    # Validate the input before continuing with later processing.
    if date_time_match is None:
        raise ValueError(f'Unexpected German date/time: {date_time_text!r}.')
    date_text, time_text = date_time_match.groups()
    local_time = datetime.strptime(f'{date_text} {time_text}', '%d.%m.%Y %H:%M').replace(
        tzinfo=BERLIN_TIMEZONE
    )
    opponent = TEAM_ALIASES.get(opponent_name)
    # Validate the input before continuing with later processing.
    if opponent is None:
        raise ValueError(f'Unknown LigaInsider opponent alias: {opponent_name!r}.')
    return ('home' if german_side == 'Heimspiel' else 'away', local_time.isoformat(timespec='minutes'), opponent)


# Handle children with class for reuse in the workflow.
def direct_children_with_class(element: Any, class_name: str) -> list[Any]:
    return [
        child for child in element.find_all('div', recursive=False)
        if class_name in child.get('class', [])
    ]


# Resolve position for reuse in the workflow.
def resolve_position(row_number: int, columns: list[Any], column_index: int) -> str:
    column_count = len(columns)
    classes = set(columns[column_index].get('class', []))
    modifiers = classes - {'player_position_column', 'mx-auto', 'd-block'}
    # Validate the input before continuing with later processing.
    if row_number == 1:
        if column_count == 1 and modifiers == set():
            return 'GK'
    # Validate the input before continuing with later processing.
    elif row_number == 2:
        # Validate the input before continuing with later processing.
        if modifiers:
            raise ValueError(f'Unexpected row-2 modifiers: {sorted(modifiers)}.')
        positions = {3: ['DCR', 'DC', 'DCL'], 4: ['DR', 'DCR', 'DCL', 'DL']}.get(column_count)
        if positions is not None:
            return positions[column_index]
    # Choose the appropriate path for the current data state.
    elif row_number == 3:
        if column_count == 5:
            positions = ['MR', 'MCR', 'DMC', 'MCL', 'ML']
            return positions[column_index]
        down_indices = [
            index for index, column in enumerate(columns)
            if 'player_position_down' in column.get('class', [])
        ]
        up_indices = [
            index for index, column in enumerate(columns)
            if 'player_position_up' in column.get('class', [])
        ]
        if column_count == 3 and len(down_indices) == 1 and len(up_indices) == 2:
            if column_index == down_indices[0]:
                return 'DMC'
            return ['MCR', 'MCL'][up_indices.index(column_index)]
        if column_count == 3 and len(down_indices) == 2 and len(up_indices) == 1:
            if column_index in down_indices:
                return ['MCR', 'MCL'][down_indices.index(column_index)]
            return 'AMC'
        if 'player_position_up_extra' in modifiers:
            return 'AMC'
        if 'player_position_down_extra' in modifiers:
            return 'DMC'
        if 'player_position_down' in modifiers:
            if column_count == 4:
                return ['MR', 'MCR', 'MCL', 'ML'][column_index]
            if column_count == 3 and len(down_indices) == 2:
                return ['MCR', 'MCL'][down_indices.index(column_index)]
        if 'player_position_up' in modifiers:
            if column_index == 0:
                return 'MR'
            if column_index == column_count - 1:
                return 'ML'
            if column_index < column_count / 2:
                return 'AMR'
            if column_index > column_count / 2:
                return 'AML'
    elif row_number == 4:
        has_smallmargin = 'smallmargin' in modifiers
        if has_smallmargin and 'player_position_down_extra' in modifiers:
            if column_count == 3 and column_index in (0, 2):
                return ['AMCR', 'AMCL'][0 if column_index == 0 else 1]
        if has_smallmargin and 'player_position_up_extra' in modifiers:
            return 'FW'
        if has_smallmargin and modifiers == {'smallmargin'} and column_count == 2:
            return ['FWCR', 'FWCL'][column_index]
        if not modifiers and column_count == 2:
            return ['FWCR', 'FWCL'][column_index]
        if 'player_position_down' in modifiers or 'player_position_down_extra' in modifiers:
            if column_index == 0:
                return 'FWR'
            if column_index == column_count - 1:
                return 'FWL'
        if 'player_position_up' in modifiers or 'player_position_up_extra' in modifiers:
            return 'FW'
    raise ValueError(
        'Unrecognised LigaInsider formation position: '
        f'row={row_number}, columns={column_count}, index={column_index}, classes={sorted(classes)}.'
    )


# Parse and validate player for reuse in the workflow.
def parse_player(player_element: Any, position: str, row_number: int, slot_index: int, probability_rank: int) -> dict[str, Any]:
    link = player_element.select_one('.player_name a[href]')
    displayed_name = clean_text(link.get_text(' ', strip=True) if link else None)
    relative_url = clean_text(link.get('href') if link else None)
    # Validate the input before continuing with later processing.
    if displayed_name is None or relative_url is None:
        raise ValueError('A formation player is missing its name or URL.')
    player_url = urljoin(SOURCE_URL, relative_url)
    injury_status = 'QUES' if player_element.select_one('.bottom_icon img[title="Angeschlagen"]') else None
    return {
        'full_name': full_name_from_player_url(player_url),
        'displayed_name': displayed_name,
        'position': position,
        'injury_status': injury_status,
        'player_id': player_id_from_player_url(player_url),
        'player_url': player_url,
        'formation_row': row_number,
        'slot_index': slot_index,
        'starting_probability_rank': probability_rank,
    }


# Parse and validate team page for reuse in the workflow.
def parse_team_page(html: str, team_name: str, team_url: str) -> dict[str, Any] | None:
    soup = BeautifulSoup(html, 'html.parser')
    heading = soup.find('h1')
    heading_text = clean_text(heading.get_text(' ', strip=True) if heading else None)
    if heading_text is None or REQUIRED_HEADING not in heading_text.upper():
        return None
    fixture_element = soup.select_one('div.team_box_right p')
    fixture_text = clean_text(fixture_element.get_text(' ', strip=True) if fixture_element else None)
    # Validate the input before continuing with later processing.
    if fixture_text is None:
        raise ValueError(f'{team_name}: missing fixture text.')
    side, match_time, opponent = parse_fixture_text(fixture_text)
    stadiums = soup.select('div.stadium_container_bg')
    # Validate the input before continuing with later processing.
    if len(stadiums) != 1:
        raise ValueError(f'{team_name}: expected one stadium container, found {len(stadiums)}.')
    rows = stadiums[0].select('div.player_position_row')
    # Validate the input before continuing with later processing.
    if len(rows) != 4:
        raise ValueError(f'{team_name}: expected four formation rows, found {len(rows)}.')
    players = []
    # Process each available item while preserving the current workflow state.
    for row_number, row in enumerate(rows, start=1):
        columns = direct_children_with_class(row, 'player_position_column')
        # Validate the input before continuing with later processing.
        if not columns:
            raise ValueError(f'{team_name}: row {row_number} has no position columns.')
        # Process each available item while preserving the current workflow state.
        for column_index, column in enumerate(columns):
            position = resolve_position(row_number, columns, column_index)
            candidates = direct_children_with_class(column, 'sub_child') or [column]
            # Process each available item while preserving the current workflow state.
            for probability_rank, candidate in enumerate(candidates, start=1):
                players.append(
                    parse_player(candidate, position, row_number, column_index + 1, probability_rank)
                )
    return {
        'team_name': team_name,
        'team_url': team_url,
        'side': side,
        'match_time': match_time,
        'opponent': opponent,
        'lineup_status': 'Predicted Lineup',
        'players': players,
    }

## 4. Load every team page through one Chrome session

In [4]:
scrape_started_at = datetime.now().astimezone()
accepted_teams = []
skipped_teams = []
driver = None
# Handle expected failures with a clear, actionable message.
try:
    driver = uc.Chrome(version_main=CHROME_MAJOR_VERSION)
    driver.set_window_size(1920, 1080)
    driver.set_page_load_timeout(PAGE_LOAD_TIMEOUT_SECONDS)
    wait = WebDriverWait(driver, WAIT_TIMEOUT_SECONDS)
    # Process each available item while preserving the current workflow state.
    for team_name, team_url in TEAM_PAGES.items():
        driver.get(team_url)
        wait.until(lambda current_driver: current_driver.execute_script('return document.readyState') == 'complete')
        wait.until(EC.presence_of_element_located((By.TAG_NAME, 'h1')))
        # Handle expected failures with a clear, actionable message.
        try:
            parsed_team = parse_team_page(driver.page_source, team_name, team_url)
        except ValueError as exc:
            raise ValueError(f'{team_name} ({team_url}): {exc}') from exc
        # Choose the appropriate path for the current data state.
        if parsed_team is None:
            skipped_teams.append(team_name)
        else:
            accepted_teams.append(parsed_team)
except TimeoutException as exc:
    raise RuntimeError('Timed out while loading a LigaInsider team page.') from exc
except WebDriverException as exc:
    raise RuntimeError(
        'Could not load LigaInsider through undetected Chrome 150. '
        'Confirm that compatible Chrome 150 is installed.'
    ) from exc
finally:
    if driver is not None:
        # Handle expected failures with a clear, actionable message.
        try:
            driver.quit()
            driver.quit = lambda: None
        except Exception as shutdown_error:
            print(f'Chrome shutdown warning: {type(shutdown_error).__name__}: {shutdown_error}')
        finally:
            driver = None
scrape_finished_at = datetime.now().astimezone()
print(f'Accepted predicted lineups: {len(accepted_teams)}; skipped: {len(skipped_teams)}')

Accepted predicted lineups: 18; skipped: 0


## 5. Assemble complete fixtures and validate the snapshot

In [5]:
fixture_sides = {}
# Process each available item while preserving the current workflow state.
for team in accepted_teams:
    home_team = team['team_name'] if team['side'] == 'home' else team['opponent']
    away_team = team['opponent'] if team['side'] == 'home' else team['team_name']
    fixture_key = (team['match_time'], home_team, away_team)
    sides = fixture_sides.setdefault(fixture_key, {})
    # Validate the input before continuing with later processing.
    if team['side'] in sides:
        raise ValueError(f'Duplicate {team["side"]} lineup for {fixture_key}.')
    sides[team['side']] = team

matches = []
unmatched_teams = []
# Process each available item while preserving the current workflow state.
for fixture_key, sides in fixture_sides.items():
    if set(sides) != {'home', 'away'}:
        unmatched_teams.extend(team['team_name'] for team in sides.values())
        continue
    match_time, home_name, away_name = fixture_key
    home, away = sides['home'], sides['away']
    # Validate the input before continuing with later processing.
    if home['opponent'] != away_name or away['opponent'] != home_name:
        raise ValueError(f'Inconsistent opponents for {fixture_key}.')
    matches.append({
        'match_time': match_time,
        'home': {key: home[key] for key in ('team_name', 'side', 'lineup_status', 'players')},
        'away': {key: away[key] for key in ('team_name', 'side', 'lineup_status', 'players')},
    })
matches.sort(key=lambda match: (match['match_time'], match['home']['team_name']))
team_count = sum(2 for _ in matches)
player_count = sum(
    len(match[side]['players']) for match in matches for side in ('home', 'away')
)
output_data = {
    'metadata': {
        'source': SOURCE_NAME,
        'source_url': SOURCE_URL,
        'captured_at': scrape_started_at.isoformat(timespec='seconds'),
        'capture_finished_at': scrape_finished_at.isoformat(timespec='seconds'),
        'match_count': len(matches),
        'team_count': team_count,
        'player_count': player_count,
        'team_page_count': len(TEAM_PAGES),
        'team_pages': TEAM_PAGES,
        'skipped_teams': skipped_teams,
        'unmatched_teams': sorted(unmatched_teams),
    },
    'matches': matches,
}
print(f'Complete fixtures: {len(matches)}; teams: {team_count}; players including alternatives: {player_count}')
if unmatched_teams:
    print(f'Unmatched accepted teams excluded from JSON: {sorted(unmatched_teams)}')

Complete fixtures: 9; teams: 18; players including alternatives: 247


## 6. Write the JSON snapshot and report results

In [6]:
filename_timestamp = scrape_started_at.strftime('%Y%m%d_%H%M%S')
output_path = output_directory / f'ligainsider_bundesliga_lineups_{filename_timestamp}.json'
temporary_output_path = output_path.with_suffix('.json.tmp')
# Handle expected failures with a clear, actionable message.
try:
    # Use the resource only within this controlled scope.
    with temporary_output_path.open('w', encoding='utf-8', newline='\n') as file:
        json.dump(output_data, file, ensure_ascii=False, indent=2)
        file.write('\n')
    temporary_output_path.replace(output_path)
except OSError as exc:
    raise OSError(f'Could not write LigaInsider snapshot {output_path}: {exc}') from exc
finally:
    temporary_output_path.unlink(missing_ok=True)

# Process each available item while preserving the current workflow state.
for match_number, match in enumerate(matches, start=1):
    print(
        f'{match_number:>2}. {match["match_time"]} | '
        f'{match["home"]["team_name"]} vs {match["away"]["team_name"]}'
    )
flagged_players = [
    f'{match[side]["team_name"]}: {player["displayed_name"]} ({player["position"]})'
    for match in matches
    for side in ('home', 'away')
    for player in match[side]['players']
    if player['injury_status'] == 'QUES'
]
print(f'\nQuestionable players: {len(flagged_players)}')
# Process each available item while preserving the current workflow state.
for player in flagged_players:
    print(f'- {player}')
print(f'\nSaved LigaInsider lineup snapshot: {output_path.resolve()}')

 1. 2026-08-28T20:30+02:00 | Bayern München vs VfB Stuttgart
 2. 2026-08-29T15:30+02:00 | 1. FC Köln vs TSG Hoffenheim
 3. 2026-08-29T15:30+02:00 | 1. FC Union Berlin vs Eintracht Frankfurt
 4. 2026-08-29T15:30+02:00 | FSV Mainz 05 vs SC Paderborn
 5. 2026-08-29T15:30+02:00 | RB Leipzig vs Borussia Mönchengladbach
 6. 2026-08-29T15:30+02:00 | SV 07 Elversberg vs Bayer 04 Leverkusen
 7. 2026-08-29T18:30+02:00 | Borussia Dortmund vs Hamburger SV
 8. 2026-08-30T15:30+02:00 | SC Freiburg vs SV Werder Bremen
 9. 2026-08-30T17:30+02:00 | FC Augsburg vs Schalke 04

Questionable players: 13
- VfB Stuttgart: Führich (FWL)
- TSG Hoffenheim: Bernardo (DL)
- Eintracht Frankfurt: Dōan (MR)
- Eintracht Frankfurt: Larsson (MCR)
- Eintracht Frankfurt: Burkardt (FWCL)
- RB Leipzig: Rômulo (FW)
- Borussia Dortmund: Bensebaini (DCL)
- Hamburger SV: Capaldo (DCR)
- Hamburger SV: Torunarigha (DCL)
- Hamburger SV: Muheim (ML)
- SV Werder Bremen: Hein (GK)
- FC Augsburg: Gouweleeuw (DC)
- Schalke 04: Karaman

In [7]:
from project_paths import prune_timestamped_outputs

removed_outputs = prune_timestamped_outputs()
print(f"Pruned {len(removed_outputs)} expired timestamped output(s).")


Pruned 0 expired timestamped output(s).
